## 00. Quick Start


In [1]:
print('Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.')
print('LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.')

Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.
LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.


## 01. Environment


In [2]:
import os, sys, json
from pathlib import Path
from IPython.display import display
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
AI_ROOT = next((p for p in SEARCH_ROOTS if (p / 'app').is_dir()), None)
if AI_ROOT is None: AI_ROOT = next((p / 'ai' for p in SEARCH_ROOTS if (p / 'ai' / 'app').is_dir()))
if str(AI_ROOT) not in sys.path: sys.path.insert(0, str(AI_ROOT))
print({'python': sys.version.split()[0], 'aiRoot': str(AI_ROOT)})

{'python': '3.14.5', 'aiRoot': 'C:\\Users\\seewo\\Desktop\\big_proj_01\\new_3\\ai'}


## 02. MODE


In [3]:
MODE = 'LIVE'  # MOCK | REPLAY | LIVE
RECORDINGS_DIR = AI_ROOT / 'recordings' / 'concept_portfolio_v2'
print({'mode': MODE, 'liveExternalOperationsEnabled': MODE == 'LIVE'})

{'mode': 'LIVE', 'liveExternalOperationsEnabled': True}


## 03. Environment Check


In [4]:
LIVE_ENV_KEYS = ['AI_PROVIDER', 'AI_API_KEY', 'AI_MODEL', 'MOLEG_API_KEY', 'LEGAL_REGISTRY_VERSION']
env_status = {key: bool(os.getenv(key)) for key in LIVE_ENV_KEYS}
print(env_status if MODE == 'LIVE' else {'mode': MODE, 'message': '외부 환경변수 불필요'})

{'AI_PROVIDER': True, 'AI_API_KEY': True, 'AI_MODEL': True, 'MOLEG_API_KEY': True, 'LEGAL_REGISTRY_VERSION': True}


## 04. Schema Preflight


In [5]:
from app.concept_portfolio_v2 import ConceptPortfolioEngine, ProviderGateway, ProviderMode
from app.concept_portfolio_v2.adapters import CurrentLegalAdapter
from app.concept_portfolio_v2.diagnostics.notebook_view import *
gateway = ProviderGateway(MODE, recordings_dir=RECORDINGS_DIR)
engine = ConceptPortfolioEngine(MODE, gateway=gateway)
schema_preflight = engine.schema_preflight_report()
display(show_schema_preflight(schema_preflight))
assert schema_preflight.status == 'PASS' and schema_preflight.providerCalls == 0

,스키마,상태,실패,Provider 호출
0,PlanDraftPool,PASS,[],0
1,ConceptCandidateDraft,PASS,[],0
2,SemanticDistinctnessResult,PASS,[],0
3,SemanticFidelityResult,PASS,[],0
4,SemanticArchitectureBatch,PASS,[],0
5,SemanticHypothesisBatch,PASS,[],0


## 05. Input


In [6]:
SCENARIO_FILE = AI_ROOT / 'fixtures' / 'concept_portfolio_v2' / 'live_scenarios.json'
SCENARIOS = {item['scenarioId']: item for item in json.loads(SCENARIO_FILE.read_text(encoding='utf-8'))}
LIVE_SCENARIO = 'AI_INTERVIEW_COACH'
LIVE_TEST_LEVEL = 'ONE_CLICK'  # CORE | LEGAL_C1 | FULL_E2E | ONE_CLICK
RUN_STAGED_CORE = LIVE_TEST_LEVEL in {'CORE', 'LEGAL_C1', 'FULL_E2E'}
RUN_STAGED_LEGAL = LIVE_TEST_LEVEL in {'LEGAL_C1', 'FULL_E2E'}
RUN_STAGED_FULL = LIVE_TEST_LEVEL == 'FULL_E2E'
scenario = SCENARIOS[LIVE_SCENARIO]
TEST_INPUT = {key: scenario[key] for key in ('ideaOverview', 'problem', 'targetUsers')}
MAX_CONCEPTS = 5
display({'scenario': LIVE_SCENARIO, 'testLevel': LIVE_TEST_LEVEL, 'domain': scenario['domain'],
         'expectedStructuralFeatures': scenario['expectedStructuralFeatures']})

{'scenario': 'AI_INTERVIEW_COACH',
 'testLevel': 'ONE_CLICK',
 'domain': 'Digital education',
 'expectedStructuralFeatures': ['순수 디지털', '불필요한 자격·파트너 미생성']}

## 06. Idea Brief Derivation


In [7]:
seed = engine.seed_adapter.adapt(TEST_INPUT)
idea_context = None
if RUN_STAGED_CORE:
    engine._reset()
    idea_context = await engine.derive_idea_brief(seed)
print({'ideaCallComplete': bool(idea_context), 'interpretationPresent': bool(seed.interpretation),
       'stagedCore': RUN_STAGED_CORE})

{'ideaCallComplete': True, 'interpretationPresent': True}


## 07. Safety


In [8]:
display(idea_context.safetyReview.model_dump(mode='json') if idea_context else {'status': 'SKIPPED'})
assert idea_context is None or idea_context.safetyReview.passed

{'decision': 'ALLOW',
 'categories': [],
 'restrictions': [],
 'userFacingReason': '이 아이디어는 취업 준비생을 위한 AI 서비스로, 면접 준비를 지원하는 안전한 서비스입니다.'}

## 08. AI가 이해한 아이디어


In [9]:
display(show_idea_interpretation(idea_context) if idea_context else {'status': 'SKIPPED'})

,항목,AI 이해 결과
0,interpretedProblem,취업 준비생은 면접 전 자신의 답변 품질과 개선점을 객관적으로 확인하기 어려운 문제...
1,interpretedTargetUsers,신입 및 주니어 채용을 준비하는 성인 취업 준비생들입니다.
2,usageContext,AI 서비스를 통해 모의면접 답변을 분석하고 피드백을 제공받는 상황입니다.
3,industryCategory,교육/취업 지원
4,researchScope,AI 기반 면접 준비 서비스
5,conciseIdeaDefinition,AI를 활용하여 취업 준비생의 면접 답변을 분석하고 개선 피드백을 제공하는 서비스입니다.
6,targetRegionInterpretation,
7,relevantKnownCompetitorContext,


## 09. Readiness / Summary / commitments


In [10]:
display(show_idea_readiness(idea_context) if idea_context else {'status': 'SKIPPED'})

{'readiness': {'status': 'READY_FOR_REVIEW',
  'score': 0,
  'missingFieldKeys': []},
 'readinessDiagnostic': 'READINESS_INCONSISTENT',
 'userFacingSummary': '이 서비스는 취업 준비생의 모의면접 답변을 분석하고 직무별 개선 피드백과 반복 연습 계획을 제공하는 AI 기반의 솔루션입니다.',
 'commitmentCandidates': [],
 'contradictions': [],
 'questions': []}

## 10. Seed Analysis


In [11]:
analysis = await engine.analyze_seed(seed) if RUN_STAGED_CORE else None
display(show_seed_analysis(analysis) if analysis else {'status': 'SKIPPED'})

,구분,값
0,탐색 폭,EXPLORE
1,다양성 수용량,5
2,설명,선택 입력 LOCK 0개로 11개 설계 차원이 열려 있습니다. diversityCa...


## 11. Generic Opportunity Kernel


In [12]:
display(analysis.opportunityKernel.model_dump(mode='json') if analysis else {'status': 'SKIPPED'})

{'problemCore': '취업 준비생은 면접 전 자신의 답변 품질과 개선점을 객관적으로 확인하기 어려운 문제를 겪고 있습니다.',
 'targetCore': '신입 및 주니어 채용을 준비하는 성인 취업 준비생들입니다.',
 'useContexts': ['AI 서비스를 통해 모의면접 답변을 분석하고 피드백을 제공받는 상황입니다.'],
 'intentComponents': ['AI를 활용하여 취업 준비생의 면접 답변을 분석하고 개선 피드백을 제공하는 서비스입니다.'],
 'mustPreserve': ['취업 준비생은 면접 전 자신의 답변 품질과 개선점을 객관적으로 확인하기 어려운 문제를 겪고 있습니다.',
  '신입 및 주니어 채용을 준비하는 성인 취업 준비생들입니다.',
  'AI를 활용하여 취업 준비생의 면접 답변을 분석하고 개선 피드백을 제공하는 서비스입니다.'],
 'maySpecialize': ['핵심 대상의 의미 있는 하위 세그먼트',
  '핵심 사용 맥락의 구체화',
  '가치 제안 또는 offer의 구체화'],
 'forbiddenDriftSummary': '핵심 문제와 대상이 모두 무관한 기회로 교체되면 범위를 벗어납니다.'}

## 12. Design Space


In [13]:
display(show_design_space(analysis) if analysis else {'status': 'SKIPPED'})

,분류,필드,값
0,SOURCE_LOCK,ideaOverview,취업 준비생의 모의면접 답변을 분석하고 직무별 개선 피드백과 반복 연습 계획을 제공...
1,SOURCE_LOCK,problem,취업 준비생은 실제 면접 전 자신의 답변 품질과 개선점을 객관적으로 확인하기 어렵다.
2,SOURCE_LOCK,targetUsers,신입·주니어 채용을 준비하는 성인 취업 준비생
3,SEMANTIC_ANCHOR,ideaOverview,취업 준비생의 모의면접 답변을 분석하고 직무별 개선 피드백과 반복 연습 계획을 제공...
4,SEMANTIC_ANCHOR,problem,취업 준비생은 실제 면접 전 자신의 답변 품질과 개선점을 객관적으로 확인하기 어렵다.
5,SEMANTIC_ANCHOR,targetUsers,신입·주니어 채용을 준비하는 성인 취업 준비생
6,OPEN,solutionMechanism,변경 가능
7,OPEN,valueDelivery,변경 가능
8,OPEN,operatingModel,변경 가능
9,OPEN,supplyStructure,변경 가능


## 13. Generate and Adaptively Replenish Plan Pool


In [14]:
plan_validation = (await engine.prepare_portfolio_plans(seed, analysis, max_concepts=MAX_CONCEPTS)
                   if RUN_STAGED_CORE else None)
plans = engine._last_plan_pool if plan_validation else []
print({'totalPlans': len(plans),
       'planningRounds': plan_validation.planningRounds if plan_validation else 0,
       'replenishmentRequested': plan_validation.replenishmentRequested if plan_validation else 0})

{'totalPlans': 14, 'planningRounds': 3, 'replenishmentRequested': 8}


## 14. Plan Count / Adaptive Replenishment Check


In [15]:
display(show_plan_pool_status(engine._last_plan_pool_status) if plan_validation else {'status': 'SKIPPED'})
display({'planningRounds': plan_validation.planningRounds if plan_validation else 0,
         'replenishmentRequested': plan_validation.replenishmentRequested if plan_validation else 0,
         'adaptiveReplenishmentUsed': bool(plan_validation and plan_validation.planningRounds > 1)})

,requestedPoolSize,returnedPoolSize,initialTarget,reserveTarget,reserveAvailable,status
0,7,6,5,2,0,RESERVE_SHORTFALL


{'planningRounds': 3,
 'replenishmentRequested': 8,
 'adaptiveReplenishmentUsed': True}

## 15. Korean Plan Display


In [16]:
display(show_portfolio_plans(plan_validation.acceptedPlans + plan_validation.reservePlans)
        if plan_validation else {'status': 'SKIPPED'})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P2,모의면접 개선 플랫폼,SELECTED,0.7992,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,기타 역할 · 기타 운영,신입 및 주니어 채용을 준비하는 성인 취업 준비생,AI 서비스를 통해 모의면접 답변을 분석하고 피드백을 제공받는 상황,AI의 분석을 통해 면접 준비생이 자신감을 가지고 면접에 임할 수 있도록 지원합니다.,"AI 분석을 통해 면접 준비생의 답변을 개선하고, 성공적인 면접을 위한 전략을 제공...",AI가 제공하는 피드백으로 면접 준비생의 경쟁력을 높입니다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",AI 기술을 활용하여 취업 준비생의 면접 준비를 혁신하는 서비스입니다.
1,P1,AI 면접 피드백 서비스,SELECTED,0.6926,같은 Family를 허용하면서 의미 있는 target/use/value Varian...,VARIANT,기타 역할 · 기타 운영,신입 및 주니어 채용을 준비하는 성인 취업 준비생,AI 서비스를 통해 모의면접 답변을 분석하고 피드백을 제공받는 상황,AI 기반의 분석을 통해 취업 준비생이 면접 준비를 보다 효과적으로 할 수 있도록 ...,"AI 기술을 활용하여 면접 답변의 질을 높이고, 개인 맞춤형 피드백을 제공합니다.",AI가 제공하는 데이터 기반 피드백으로 취업 준비생의 면접 준비를 혁신합니다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",AI 기술을 활용하여 취업 준비생의 면접 준비를 혁신하는 서비스입니다.


## 16. Plan Lock/Intent Validation


In [17]:
display({'accepted': [p.planId for p in plan_validation.acceptedPlans] if plan_validation else [],
         'rejected': [p.model_dump(mode='json') for p in plan_validation.rejectedPlans]
                     if plan_validation else []})

{'accepted': ['P2', 'P1'],
 'rejected': [{'planId': 'P3',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P2'},
  {'planId': 'P4',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P2'},
  {'planId': 'P5',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P2'},
  {'planId': 'P6',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P2'},
  {'planId': 'P7',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P1'},
  {'planId': 'P8',
   'reasonCode': 'PLAN_DUPLICATE',
   'safeSummary': 'Concept Thesis와 Architecture가 기존 Plan과 사실상 같습니다.',
   'conflictPlanId': 'P2'},
  {'planId': 'P9',
   'reasonCode': 'PLAN_DUPLICA

## 17. Portfolio Family / Variant / Distinct


In [18]:
display(show_plan_diversity(plan_validation.diversity) if plan_validation else {'status': 'SKIPPED'})

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,P1,P2,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
1,P1,P3,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
2,P2,P3,DUPLICATE,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...",,THESIS_AND_ARCHITECTURE,False,이름이나 표현을 제외한 Concept Thesis와 Business Architec...
3,P1,P4,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
4,P2,P4,DUPLICATE,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...",,THESIS_AND_ARCHITECTURE,False,이름이나 표현을 제외한 Concept Thesis와 Business Architec...
5,P1,P5,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
6,P2,P5,DUPLICATE,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...",,THESIS_AND_ARCHITECTURE,False,이름이나 표현을 제외한 Concept Thesis와 Business Architec...
7,P1,P6,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
8,P2,P6,DUPLICATE,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...",,THESIS_AND_ARCHITECTURE,False,이름이나 표현을 제외한 Concept Thesis와 Business Architec...
9,P1,P7,DUPLICATE,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...",,THESIS_AND_ARCHITECTURE,False,이름이나 표현을 제외한 Concept Thesis와 Business Architec...


## 18. Selected + Reserve Plans


In [19]:
selected_plans = plan_validation.acceptedPlans if plan_validation else []
reserve_plans = plan_validation.reservePlans if plan_validation else []
display(show_portfolio_plans(selected_plans + reserve_plans))
display({'selected': [p.planId for p in selected_plans], 'reserve': [p.planId for p in reserve_plans]})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P2,모의면접 개선 플랫폼,SELECTED,0.7992,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,기타 역할 · 기타 운영,신입 및 주니어 채용을 준비하는 성인 취업 준비생,AI 서비스를 통해 모의면접 답변을 분석하고 피드백을 제공받는 상황,AI의 분석을 통해 면접 준비생이 자신감을 가지고 면접에 임할 수 있도록 지원합니다.,"AI 분석을 통해 면접 준비생의 답변을 개선하고, 성공적인 면접을 위한 전략을 제공...",AI가 제공하는 피드백으로 면접 준비생의 경쟁력을 높입니다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",AI 기술을 활용하여 취업 준비생의 면접 준비를 혁신하는 서비스입니다.
1,P1,AI 면접 피드백 서비스,SELECTED,0.6926,같은 Family를 허용하면서 의미 있는 target/use/value Varian...,VARIANT,기타 역할 · 기타 운영,신입 및 주니어 채용을 준비하는 성인 취업 준비생,AI 서비스를 통해 모의면접 답변을 분석하고 피드백을 제공받는 상황,AI 기반의 분석을 통해 취업 준비생이 면접 준비를 보다 효과적으로 할 수 있도록 ...,"AI 기술을 활용하여 면접 답변의 질을 높이고, 개인 맞춤형 피드백을 제공합니다.",AI가 제공하는 데이터 기반 피드백으로 취업 준비생의 면접 준비를 혁신합니다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",AI 기술을 활용하여 취업 준비생의 면접 준비를 혁신하는 서비스입니다.


{'selected': ['P2', 'P1'], 'reserve': []}

## 19. Candidate 1


In [20]:
candidate_one = (await engine.expand_plan(seed, selected_plans[0], 1)
                 if RUN_STAGED_CORE and selected_plans else None)
display(show_candidates([candidate_one]) if candidate_one else [])

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,모의면접 개선 플랫폼,"AI 알고리즘을 통해 면접 답변을 분석하고, 개선점을 도출하여 피드백을 제공합니다.",기타 역할 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '신입·주니어 채용을...,구독 모델을 통한 지속적인 서비스 제공,AI 분석 시스템과 사용자 인터페이스를 통해 운영됩니다.


## 20. Candidate 1 Korean/Governance


In [21]:
candidate_one_reports = []
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'status': 'PENDING_FULL_CANDIDATE_RECOVERY'})

{'candidateId': 'C1', 'status': 'PENDING_FULL_CANDIDATE_RECOVERY'}

## 21. Candidate 1 Actual Generic Descriptor


In [22]:
display(show_concept_descriptors([candidate_one]) if candidate_one else [])

,entityId,family,dimension,code,confidence,source
0,C1,OTHER:PARTNER_NETWORK,businessRole,OTHER,LOW,UNKNOWN
1,C1,OTHER:PARTNER_NETWORK,operatingModel,PARTNER_NETWORK,HIGH,RULE
2,C1,OTHER:PARTNER_NETWORK,partnerModel,PARTNER_NETWORK,HIGH,RULE
3,C1,OTHER:PARTNER_NETWORK,deliveryModel,DIGITAL,HIGH,RULE
4,C1,OTHER:PARTNER_NETWORK,transactionModel,OTHER,LOW,UNKNOWN
5,C1,OTHER:PARTNER_NETWORK,monetizationModel,OTHER,LOW,UNKNOWN
6,C1,OTHER:PARTNER_NETWORK,customerInteractionModel,APP,HIGH,RULE
7,C1,OTHER:PARTNER_NETWORK,dataDependency,MATERIAL,NaN,NaN
8,C1,OTHER:PARTNER_NETWORK,physicalDependency,MATERIAL,NaN,NaN


## 22. Candidate 1 Fidelity


In [23]:
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'})

{'candidateId': 'C1',
 'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'}

## 23. Remaining Candidates


In [24]:
remaining_candidates = []
if RUN_STAGED_CORE:
    for i, plan in enumerate(selected_plans[1:], 2):
        remaining_candidates.append(await engine.expand_plan(seed, plan, i))
candidate_drafts = ([candidate_one] if candidate_one else []) + remaining_candidates
display(show_candidates(candidate_drafts))

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,모의면접 개선 플랫폼,"AI 알고리즘을 통해 면접 답변을 분석하고, 개선점을 도출하여 피드백을 제공합니다.",기타 역할 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '신입·주니어 채용을...,구독 모델을 통한 지속적인 서비스 제공,AI 분석 시스템과 사용자 인터페이스를 통해 운영됩니다.
1,C2,L2,None,AI 면접 피드백 서비스,"AI 알고리즘을 통해 면접 답변을 분석하고, 개선점을 도출하여 개인 맞춤형 피드백을...",기타 역할 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '신입·주니어 채용을...,구독 모델을 통한 지속적인 수익 창출,AI 분석 시스템과 사용자 인터페이스를 통해 운영됩니다.


## 24. Candidate Actual Generic Descriptors


In [25]:
display(show_concept_descriptors(candidate_drafts))

,entityId,family,dimension,code,confidence,source
0,C1,OTHER:PARTNER_NETWORK,businessRole,OTHER,LOW,UNKNOWN
1,C1,OTHER:PARTNER_NETWORK,operatingModel,PARTNER_NETWORK,HIGH,RULE
2,C1,OTHER:PARTNER_NETWORK,partnerModel,PARTNER_NETWORK,HIGH,RULE
3,C1,OTHER:PARTNER_NETWORK,deliveryModel,DIGITAL,HIGH,RULE
4,C1,OTHER:PARTNER_NETWORK,transactionModel,OTHER,LOW,UNKNOWN
5,C1,OTHER:PARTNER_NETWORK,monetizationModel,OTHER,LOW,UNKNOWN
6,C1,OTHER:PARTNER_NETWORK,customerInteractionModel,APP,HIGH,RULE
7,C1,OTHER:PARTNER_NETWORK,dataDependency,MATERIAL,NaN,NaN
8,C1,OTHER:PARTNER_NETWORK,physicalDependency,MATERIAL,NaN,NaN
9,C2,OTHER:PARTNER_NETWORK,businessRole,OTHER,LOW,UNKNOWN


## 25. Candidate Recovery / Portfolio Relations


In [26]:
candidate_preparation = (await engine.prepare_candidate_portfolio(
    seed, plan_validation, max_concepts=MAX_CONCEPTS, initial_candidates=candidate_drafts)
    if RUN_STAGED_CORE and plan_validation else None)
candidates = candidate_preparation.candidates if candidate_preparation else []
candidate_reports = candidate_preparation.reports if candidate_preparation else []
display(show_candidate_recovery(candidate_preparation) if candidate_preparation else {'status': 'SKIPPED'})
candidate_pairwise = [engine.compare_candidates(candidates[i], candidates[j])
                      for i in range(len(candidates)) for j in range(i + 1, len(candidates))]
display(show_plan_diversity(candidate_pairwise))

{'summary':    candidateGenerated  candidateAcceptedInitially  candidateRegenerated  \
 0                   2                           2                     0   
 
    candidateRecovered  reservePlansActivated  candidateRecoveryReplans  \
 0                   0                      0                         2   
 
    finalCandidatePortfolio  
 0                        2  ,
 'attempts':   candidateId  schemaValid  hardLockPreserved  semanticAnchorPreserved  \
 0          C1         True               True                     True   
 1          C2         True               True                     True   
 
    planFidelity anchorDecision fidelityDecision  contentLanguageValid  \
 0          True           PASS             PASS                  True   
 1          True           PASS             PASS                  True   
 
    accepted   outcome  semanticFidelityUsed  \
 0      True  ACCEPTED                  True   
 1      True  ACCEPTED                  True   
 
             

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,C1,C2,VARIANT,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,PLATFORM_INFRASTRUCTURE:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","physicalDependency, valuePropositionThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...


## 26. Legal Fact Completeness + Business Design Completion + C1 Fact Pattern


In [27]:
prechecks = [engine.legal_precheck(item) for item in candidates] if RUN_STAGED_LEGAL else []
display(show_legal_precheck(prechecks))
legal_preparation = (await engine.prepare_legal_candidates(seed, candidates)
                     if RUN_STAGED_LEGAL and candidates else None)
candidates_before_legal = candidates
candidates = legal_preparation.candidates if legal_preparation else candidates
display({'factCompleteness': [item.model_dump(mode='json') for item in legal_preparation.reports]
                              if legal_preparation else [],
         'roleSemanticBatchCalls': legal_preparation.roleSemanticBatchCalls if legal_preparation else 0,
         'completionAttempted': legal_preparation.completionAttempted if legal_preparation else 0,
         'completionValidated': legal_preparation.completionValidated if legal_preparation else 0,
         'completionAccepted': legal_preparation.completionAccepted if legal_preparation else 0,
         'completionExhausted': legal_preparation.completionExhausted if legal_preparation else 0,
         'preLegalExclusions': legal_preparation.excludedCandidates if legal_preparation else []})
display(show_legal_fact_pattern(candidates[0].candidate, seed) if candidates else [])

,candidateId,label,directSeller,intermediary,regulatedPhysicalActivity,personalDataDependency,qualificationDependency,riskHints
0,C1,Structural risk precheck — not final legal review,True,False,True,True,False,"[물리 활동, 개인정보]"
1,C2,Structural risk precheck — not final legal review,True,False,False,True,False,[개인정보]


{'factCompleteness': [{'candidateId': 'C1',
   'status': 'COMPLETABLE',
   'missingDesignFacts': ['intermediaryRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.'],
   'contradictions': [],
   'completionRequirements': ['intermediaryRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.'],
   'affectedFields': ['intermediaryRole'],
   'roleSemantics': [{'field': 'platformRole',
     'status': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'providerRole',
     'status': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'sellerRole',
     'status': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'intermediaryRole',
     'status': 'AMBIGUOUS',
     'safeReason': '해당 역할의 주체와 책임이 불명확합니다.'}],
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'},
  {'candidateId': 'C1-F1',
   'status': 'COMPLETABLE',
   'missingDesignFacts': ['intermediaryRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.'],
   'contradictions': [],
   'completionRequirements': ['intermediaryRole의 역할 존재·부재

[]

## 27. Prepared Legal C1 Evidence Summary


In [28]:
legal_adapter = CurrentLegalAdapter() if RUN_STAGED_LEGAL else None
legal_c1_input = (legal_adapter.task_input(candidates[0].candidate, seed)
                  if legal_adapter and candidates else None)
display({'candidateId': candidates[0].candidateId if candidates else None,
         'externalFacts': legal_c1_input['externalFactContext']['facts'] if legal_c1_input else [],
         'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'})

{'candidateId': None,
 'externalFacts': [],
 'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'}

## 28. Full Evidence Judgment — C1 Staged Smoke


In [29]:
RUN_FULL_LEGAL_C1 = RUN_STAGED_LEGAL
legal_one = None
if RUN_FULL_LEGAL_C1 and candidates:
    try:
        legal_one = await engine.review_legal_candidate(seed, candidates[0])
        display(show_legal_result([legal_one]))
    except Exception:
        display(show_legal_failure(candidates[0].candidateId, engine.gateway))
else:
    print('SKIPPED — RUN_FULL_LEGAL_C1=True로 명시해야 실행됩니다.')

SKIPPED — RUN_FULL_LEGAL_C1=True로 명시해야 실행됩니다.


## 29. C1 Route + Staged Redesign/Compliance/Second Legal


In [30]:
c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = ([], [], [], 0, 0)
if legal_one and candidates:
    c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[:1], [legal_one])
display({'initialRoute': legal_one.route.value if legal_one else 'SKIPPED',
         'redesignRequirements': legal_one.redesignRequirements if legal_one else [],
         'recoveryReviews': [item.model_dump(mode='json') for item in c1_legal_all[1:]],
         'requiredInputs': c1_required_inputs, 'terminalCandidates': len(c1_portfolio),
         'redesigned': c1_redesigned, 'replanned': c1_replanned})

{'initialRoute': 'SKIPPED',
 'redesignRequirements': [],
 'recoveryReviews': [],
 'requiredInputs': [],
 'terminalCandidates': 0,
 'redesigned': 0,
 'replanned': 0}

## 30. Remaining 4 Legal + Exhaustive Recovery Summary


In [31]:
RUN_REMAINING_LEGAL = RUN_STAGED_FULL
legal_remaining = []
portfolio, legal_all, required_inputs = (list(c1_portfolio), list(c1_legal_all), list(c1_required_inputs))
redesigned_count, replanned_count = c1_redesigned, c1_replanned
c1_terminal = bool(c1_portfolio or c1_required_inputs or (c1_legal_all and c1_legal_all[-1].route.value == 'SYSTEM_FAILURE'))
if RUN_REMAINING_LEGAL and c1_terminal and len(candidates) > 1:
    legal_remaining = await engine.review_legal(seed, candidates[1:])
    rest_portfolio, rest_legal, rest_inputs, rest_redesigned, rest_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[1:], legal_remaining)
    portfolio += rest_portfolio; legal_all += rest_legal; required_inputs += rest_inputs
    redesigned_count += rest_redesigned; replanned_count += rest_replanned
else:
    print('SKIPPED — C1이 정상 terminal에 도달한 후 remaining Legal을 실행합니다.')
legal_initial = ([legal_one] if legal_one else []) + legal_remaining
display({'recoveryTrace': [item.model_dump(mode='json') for item in legal_all
                           if item.candidateId not in {x.candidateId for x in legal_initial}],
         'requiredInputs': required_inputs, 'metrics': engine._legal_metrics})
print({'Plan Selected': len(selected_plans),
       'Candidate Generated': candidate_preparation.candidateGenerated if candidate_preparation else 0,
       'Candidate Valid Initially': candidate_preparation.candidateAcceptedInitially if candidate_preparation else 0,
       'Candidate Regenerated': candidate_preparation.candidateRegenerated if candidate_preparation else 0,
       'Candidate Recovered': candidate_preparation.candidateRecovered if candidate_preparation else 0,
       'Fact Completion Attempted': legal_preparation.completionAttempted if legal_preparation else 0,
       'Fact Completion Validated': legal_preparation.completionValidated if legal_preparation else 0,
       'Fact Completion Accepted': legal_preparation.completionAccepted if legal_preparation else 0,
       'Legal Ready': len(candidates) if legal_preparation else 0,
       'Legal Reviewed': len(legal_initial),
       'Legal Accepted': sum(item.route.value == 'ACCEPT' for item in legal_all),
       'Legal Redesigned': redesigned_count, 'Legal Replanned': replanned_count,
       'Final Portfolio': len(portfolio)})

SKIPPED — C1이 정상 terminal에 도달한 후 remaining Legal을 실행합니다.


{'recoveryTrace': [],
 'requiredInputs': [],
 'metrics': {'factAttempted': 2,
  'factValidated': 2,
  'factAccepted': 0,
  'factExhausted': 2,
  'redesignAttempted': 0,
  'redesignValidated': 0,
  'redesignAccepted': 0,
  'redesignExhausted': 0,
  'replanAttempted': 0,
  'replanValidated': 0,
  'replanAccepted': 0,
  'replanExhausted': 0}}

{'Plan Selected': 2, 'Candidate Generated': 2, 'Candidate Accepted': 0, 'Legal Accepted': 0, 'Legal Redesigned': 0, 'Legal Replanned': 0, 'Final Portfolio': 0}


## 31. Replan


In [32]:
display(show_replan(type('PortfolioView', (), {'concepts': portfolio})()))
print({'replanned': replanned_count, 'reserveAvailable': len(reserve_plans)})

""


{'replanned': 0, 'reserveAvailable': 0}


## 32. Final Portfolio


In [33]:
legal_terminal_status = ('READY_FULL' if len(portfolio) == MAX_CONCEPTS else
    'READY_LIMITED' if portfolio else
    'LEGAL_RECOVERY_COMPLETE_NO_ACCEPTED_CANDIDATE' if legal_initial and len(legal_initial) == len(candidates)
    else 'LEGAL_PENDING')
display(show_final_portfolio(type('PortfolioView', (), {'concepts': portfolio})()) if portfolio else {'status': legal_terminal_status})

{'status': 'LEGAL_PENDING'}

## 33. Unresolved Candidate Summary


In [34]:
display(show_required_inputs(required_inputs) if required_inputs else {'unresolved': []})

{'unresolved': []}

## 34. Manual Concept Selection


In [35]:
SELECTED_CANDIDATE_ID = portfolio[0].candidateId if portfolio else None  # 사용자가 수정
selected_concept = next((item for item in portfolio if item.candidateId == SELECTED_CANDIDATE_ID), None)
print({'selectedCandidateId': SELECTED_CANDIDATE_ID})

{'selectedCandidateId': None}


## 35. 7 Hypotheses


In [36]:
hypotheses = (engine.build_or_load_current_hypothesis_contract(selected_concept)
              if RUN_STAGED_FULL and selected_concept else [])
hypotheses = await engine.resolve_hypothesis_semantics(hypotheses) if hypotheses else []
display(show_hypotheses(hypotheses))
display(show_hypothesis_readiness(hypotheses))

""


{'All Hypotheses Semantically Ready': True,
 'status': 'READY',
 'reason': None,
 'unresolvedHypotheses': []}

## 36. Confirm / Edit


In [37]:
CONFIRM_ALL_PROPOSED = True
HYPOTHESIS_EDITS = {
    # 'PRICE': '월 17,900원',
}
confirmed_hypotheses = engine.confirm_hypotheses(
    hypotheses, HYPOTHESIS_EDITS, confirm_all_proposed=CONFIRM_ALL_PROPOSED) if hypotheses else []
display(show_hypotheses(confirmed_hypotheses))
hypothesis_readiness = show_hypothesis_readiness(confirmed_hypotheses)
display(hypothesis_readiness)

""


{'All Hypotheses Semantically Ready': True,
 'status': 'READY',
 'reason': None,
 'unresolvedHypotheses': []}

## 37. Actual Delta Legal


In [38]:
RUN_DELTA_LEGAL = True
delta_legal_result = None
if RUN_STAGED_FULL and RUN_DELTA_LEGAL and selected_concept and any(h.deltaLegalRequired for h in confirmed_hypotheses):
    delta_legal_result = await engine.review_delta_legal(seed, selected_concept, confirmed_hypotheses)
    confirmed_hypotheses = engine.mark_delta_legal_reviewed(confirmed_hypotheses, delta_legal_result)
display(delta_legal_result.model_dump(mode='json') if delta_legal_result else {'status': 'NOT_REQUIRED_OR_SKIPPED'})

{'status': 'NOT_REQUIRED_OR_SKIPPED'}

## 38. Market Seed


In [39]:
handoff = None
if selected_concept and legal_all and hypothesis_readiness['Ready For Handoff']:
    handoff = engine.build_downstream_handoff(seed, selected_concept, confirmed_hypotheses, legal_all)
display(handoff.marketAnalysisSeedSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'status': 'NOT_READY', 'reason': None, 'unresolvedHypotheses': []}

## 39. Marketing Source


In [40]:
display(handoff.marketingSourceSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'status': 'NOT_READY', 'reason': None, 'unresolvedHypotheses': []}

## 40. Contract Compatibility


In [41]:
display(show_downstream_handoff(handoff) if handoff else {
    'contract': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'contract': 'NOT_READY', 'reason': None, 'unresolvedHypotheses': []}

## 41. Trace


In [42]:
display(show_trace(engine.trace))

,순서,시각,stage,action,entity,parent,status,mode,호출,요약,decision,reasonCode
0,1,2026-08-10T03:50:58.028697+00:00,CREATED,CREATED,NaN,NaN,RUNNING,LIVE,NaN,V2 Lab 실행을 생성했습니다.,NaN,NaN
1,2,2026-08-10T03:51:04.351986+00:00,SAFETY_CHECKING,IDEA_BRIEF_DERIVED,NaN,NaN,PASS,LIVE,1.0,Idea interpretation/readiness를 보존했습니다: READY_F...,NaN,NaN
2,3,2026-08-10T03:51:04.352007+00:00,SAFETY_CHECKING,READINESS_INCONSISTENT,NaN,NaN,WARNING,LIVE,NaN,READY_FOR_REVIEW이지만 score=0입니다. V2 gating은 막지 ...,READINESS_INCONSISTENT,NaN
3,4,2026-08-10T03:51:04.683645+00:00,SEED_ANALYZING,ANALYZED,lab-idea-brief,NaN,PASS,LIVE,NaN,필수 3개와 LOCK 3개를 분류했습니다.,NaN,NaN
4,5,2026-08-10T03:51:04.683922+00:00,SEED_ANALYZING,DESIGN_SPACE_READY,NaN,NaN,PASS,LIVE,NaN,Open=11 Constrained=0 Breadth=EXPLORE,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
107,108,2026-08-10T03:54:15.848901+00:00,CANDIDATE_VALIDATING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,Candidate 검사를 분리 수행합니다.,NaN,NaN
108,109,2026-08-10T03:54:19.193737+00:00,CANDIDATE_VALIDATING,ARCHITECTURE_SEMANTIC_FALLBACK,NaN,NaN,PASS,LIVE,21.0,low-confidence Candidate architecture 1개를 batc...,NaN,NaN
109,110,2026-08-10T03:54:19.194218+00:00,CANDIDATE_VALIDATING,FIDELITY_AMBIGUOUS,C2-F1,NaN,RUNNING,LIVE,NaN,lexical 판정이 불확실해 semantic fidelity를 요청합니다.,NaN,CANDIDATE_FIDELITY_RECOVERABLE
110,111,2026-08-10T03:54:21.346979+00:00,CANDIDATE_VALIDATING,SEMANTIC_FIDELITY_CHECKED,C2-F1,NaN,PASS,LIVE,22.0,AI를 활용하여 취업 준비생의 면접 답변을 분석하고 개선 피드백을 제공하는 서비스로...,PASS,NaN


## 42. Provider/Legal Usage


In [43]:
display(show_provider_usage(engine.gateway.usage))
print('상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.')

,논리 작업,상위 외부 작업,논리 stage별,상위 외부 작업 stage별,재시도,소요(ms),모드별,token,보고 비용
0,22,22,"{'SAFETY_CHECKING': 1, 'PLANNING': 5, 'NORMALI...","{'SAFETY_CHECKING': 1, 'PLANNING': 5, 'NORMALI...",0,200936,{'LIVE': 22},None,None


상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.


## 43. Replay Manifest


In [44]:
display(show_replay_manifest(engine.gateway))

{'status': 'REPLAY_PARTIAL',
 'entries':                    operation  \
 0                  PLAN_POOL   
 1                  PLAN_POOL   
 2          SEMANTIC_RELATION   
 3    NORMALIZE_ARCHITECTURES   
 4    NORMALIZE_ARCHITECTURES   
 ..                       ...   
 219                   EXPAND   
 220                 REDESIGN   
 221                   EXPAND   
 222  NORMALIZE_ARCHITECTURES   
 223    LEGAL_FACT_COMPLETION   
 
                                                   hash operationVersion  \
 0    038e1d9753b8c07ad835823495b54906a14fb6542c3c70...               v3   
 1    03b73ea7d485636893670a848e28d49e239cc8bc0c5239...             v2.1   
 2    05680e1eabb667cd53d8bbde75192650b626fd5dfaa3fd...               v3   
 3    075aa77bf976225434edecdec878de66f1b7a809f23220...               v1   
 4    07ba75e844ed104c5fc7d25bf69f296244d0925e2f8916...               v1   
 ..                                                 ...              ...   
 219  fa3806ec6a088c6605461f5c

## 44. One-click MOCK


In [45]:
mock_result = None
if LIVE_TEST_LEVEL == 'FULL_E2E' and MODE != 'LIVE':
    mock_result = await ConceptPortfolioEngine('MOCK').run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=True)
display(show_run_summary(mock_result) if mock_result else {'status': 'SKIPPED'})
assert mock_result is None or (mock_result.handoff and mock_result.handoff.contractStatus == 'CONTRACT_PASS')

,runId,runStatus,runtimeStage,producedConceptCount,downstreamReadiness,safety,requestedMaximum,planned,planSelected,planDuplicatesRemoved,...,legalRedesigned,replanned,finalPortfolio,portfolioStatus,selectedConcept,downstreamHandoff,providerCalls,totalDurationMs,failureStage,failureCode
0,14604439-7dd4-4344-bfff-ecad0276754b,READY_FULL,READY,5,PASS,PASS,5,7,5,0,...,0,0,5,READY_FULL,직접 운영 핵심형,PASS,0,188,None,None


## 45. One-click REPLAY


In [46]:
RUN_ONE_CLICK_REPLAY = False
replay_result = None
if RUN_ONE_CLICK_REPLAY:
    replay_gateway = ProviderGateway('REPLAY', recordings_dir=RECORDINGS_DIR)
    replay_result = await ConceptPortfolioEngine('REPLAY', gateway=replay_gateway).run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=False)
display(show_run_summary(replay_result) if replay_result else {'status': 'SKIPPED'})

{'status': 'SKIPPED'}

## 46. One-click LIVE


In [47]:
RUN_ONE_CLICK_LIVE = True
live_result = None
if RUN_ONE_CLICK_LIVE and LIVE_TEST_LEVEL == 'ONE_CLICK':
    assert MODE == 'LIVE', 'MODE=LIVE를 먼저 명시하세요.'
    one_click_gateway = ProviderGateway('LIVE', recordings_dir=RECORDINGS_DIR)
    one_click_engine = ConceptPortfolioEngine('LIVE', gateway=one_click_gateway)
    live_result = await one_click_engine.run_full(TEST_INPUT, max_concepts=MAX_CONCEPTS,
                                                  auto_confirm_hypotheses=False)
display(show_run_summary(live_result) if live_result else {'status': 'SKIPPED'})
if live_result:
    display(show_live_validation_summary(LIVE_SCENARIO, live_result))
    display(show_required_inputs(live_result))
    display(show_pre_legal_exclusions(live_result))
    if live_result.runStatus.value == 'FAILED':
        display(show_run_failure(live_result))
        display(show_provider_failure(one_click_engine.gateway))
        display(show_provider_usage(live_result.providerUsage))
        display(show_trace(live_result.trace[-20:]))
        display({'unresolvedCandidates': live_result.unresolvedCandidates,
                 'lastSuccessfulStage': live_result.failureDiagnostics.lastSuccessfulStage if live_result.failureDiagnostics else None,
                 'firstFailedStage': live_result.failureDiagnostics.firstFailedStage if live_result.failureDiagnostics else None})

,runId,runStatus,runtimeStage,producedConceptCount,downstreamReadiness,safety,requestedMaximum,planned,planSelected,planDuplicatesRemoved,...,legalRedesigned,replanned,finalPortfolio,portfolioStatus,selectedConcept,downstreamHandoff,providerCalls,totalDurationMs,failureStage,failureCode
0,620ac609-9e64-4296-bac8-6cd9234b3f34,FAILED,FAILED,0,INVALID,PASS,5,12,5,5,...,0,0,0,FAILED,None,INVALID,35,234166,FAILED,None


,Scenario,Plan returned,Plan selected,Candidate valid,Legal ready,Legal ACCEPT,NEEDS_INPUT,Final portfolio,Hypothesis valid,Handoff,Provider ops,Duration(ms)
0,AI_INTERVIEW_COACH,12,5,10,0,0,0,0,INVALID,PENDING,35,234166


,candidateId,scope,unknownFacts,reason,possibleUserAction,currentValue,requiredLegalChange,safeSummary
0,C1-F1,CANDIDATE,None,None,None,None,None,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.
1,C2-F1,CANDIDATE,None,None,None,None,None,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.
2,C3-F1,CANDIDATE,None,None,None,None,None,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.
3,C4-F1,CANDIDATE,None,None,None,None,None,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.
4,C5-F1,CANDIDATE,None,None,None,None,None,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.


,failedStage,failureCode,safeSummary,failedEntityId,providerFailure,lastSuccessfulStage,firstFailedStage,lastTraceEvents
0,FAILED,UNCLASSIFIED_SYSTEM_FAILURE,최종 Portfolio=0,None,{},CANDIDATE_VALIDATING,FAILED,20


{'상태': '기록된 Provider 실패 없음'}

,논리 작업,상위 외부 작업,논리 stage별,상위 외부 작업 stage별,재시도,소요(ms),모드별,token,보고 비용
0,35,35,"{'SAFETY_CHECKING': 1, 'PLANNING': 3, 'NORMALI...","{'SAFETY_CHECKING': 1, 'PLANNING': 3, 'NORMALI...",0,234062,{'LIVE': 35},None,None


,순서,시각,stage,action,entity,parent,status,mode,호출,요약,decision,reasonCode
0,1,2026-08-10T03:57:47.701310+00:00,CANDIDATE_VALIDATING,STARTED,NaN,None,RUNNING,LIVE,NaN,Candidate 검사를 분리 수행합니다.,NaN,NaN
1,2,2026-08-10T03:57:49.966652+00:00,CANDIDATE_VALIDATING,ARCHITECTURE_SEMANTIC_FALLBACK,NaN,None,PASS,LIVE,28.0,low-confidence Candidate architecture 1개를 batc...,NaN,NaN
2,3,2026-08-10T03:57:49.966970+00:00,CANDIDATE_VALIDATING,FIDELITY_AMBIGUOUS,C3-F1,None,RUNNING,LIVE,NaN,lexical 판정이 불확실해 semantic fidelity를 요청합니다.,NaN,CANDIDATE_FIDELITY_RECOVERABLE
3,4,2026-08-10T03:57:51.817502+00:00,CANDIDATE_VALIDATING,SEMANTIC_FIDELITY_CHECKED,C3-F1,None,PASS,LIVE,29.0,AI 기반의 모의면접 분석 및 피드백 제공 서비스는 정확한 피드백과 반복 연습을 통...,PASS,NaN
4,5,2026-08-10T03:57:51.818037+00:00,CANDIDATE_VALIDATING,VALIDATED,C3-F1,None,PASS,LIVE,NaN,검사 통과,NaN,NaN
5,6,2026-08-10T03:57:51.818953+00:00,LEGAL_RECOVERING,LEGAL_FACT_COMPLETENESS_CHECKED,C4,None,COMPLETABLE,LIVE,NaN,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.,COMPLETABLE,NaN
6,7,2026-08-10T03:57:59.405504+00:00,CANDIDATE_VALIDATING,STARTED,NaN,None,RUNNING,LIVE,NaN,Candidate 검사를 분리 수행합니다.,NaN,NaN
7,8,2026-08-10T03:58:02.330113+00:00,CANDIDATE_VALIDATING,ARCHITECTURE_SEMANTIC_FALLBACK,NaN,None,PASS,LIVE,31.0,low-confidence Candidate architecture 1개를 batc...,NaN,NaN
8,9,2026-08-10T03:58:02.330599+00:00,CANDIDATE_VALIDATING,FIDELITY_AMBIGUOUS,C4-F1,None,RUNNING,LIVE,NaN,lexical 판정이 불확실해 semantic fidelity를 요청합니다.,NaN,CANDIDATE_FIDELITY_RECOVERABLE
9,10,2026-08-10T03:58:05.856126+00:00,CANDIDATE_VALIDATING,SEMANTIC_FIDELITY_CHECKED,C4-F1,None,PASS,LIVE,32.0,"AI와 커뮤니티의 힘으로 면접 준비를 지원하는 플랫폼으로, AI 분석과 커뮤니티 피...",PASS,NaN


{'unresolvedCandidates': [{'candidateId': 'C1-F1',
   'scope': 'CANDIDATE',
   'reasonCode': 'LEGAL_FACT_COMPLETION_EXHAUSTED',
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'},
  {'candidateId': 'C2-F1',
   'scope': 'CANDIDATE',
   'reasonCode': 'LEGAL_FACT_COMPLETION_EXHAUSTED',
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'},
  {'candidateId': 'C3-F1',
   'scope': 'CANDIDATE',
   'reasonCode': 'LEGAL_FACT_COMPLETION_EXHAUSTED',
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'},
  {'candidateId': 'C4-F1',
   'scope': 'CANDIDATE',
   'reasonCode': 'LEGAL_FACT_COMPLETION_EXHAUSTED',
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'},
  {'candidateId': 'C5-F1',
   'scope': 'CANDIDATE',
   'reasonCode': 'LEGAL_FACT_COMPLETION_EXHAUSTED',
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'}],
 'lastSuccessfulStage': 'CANDIDATE_VALIDATING',
 'firstFailedStage': 'FAILED'}